# Financial Bank Statement Extraction — Multi-Folder Dataset with Golden Labels

This notebook demonstrates how to:
1. Read seed samples from `files/` (each folder contains a bank statement PDF + `qa-pairs.csv` golden labels)
2. Upload as a **multi-folder** seed dataset to DataFramer
3. Generate a Specification that understands the relationship between the PDFs and their Q&A pairs
4. Edit the spec to steer distributions
5. Run a generation job producing 5 synthetic samples
6. Download the output, then evaluate the first generated PDF against its Q&A pairs using OpenAI

## Setup

In [ ]:
import os

!git clone https://github.com/aimonlabs/dataframer-docs-public.git
os.chdir("dataframer-docs-public/financial-bank-text-extraction-with-golden-labels")

In [2]:
!pip install openai pymupdf pandas pydataframer tenacity pyyaml requests

import copy
import io
import os
import zipfile
from datetime import datetime
from pathlib import Path

import fitz  # PyMuPDF
import pandas as pd
import requests
import yaml
from openai import OpenAI

import dataframer
from dataframer import Dataframer
from tenacity import retry, retry_if_result, stop_never, wait_fixed

if not os.environ.get("OPENAI_API_KEY"):
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

if not os.environ.get("DATAFRAMER_API_KEY"):
    import getpass
    os.environ["DATAFRAMER_API_KEY"] = getpass.getpass("Enter DATAFRAMER_API_KEY: ")

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
df_client = Dataframer(api_key=os.environ["DATAFRAMER_API_KEY"])

FILES_DIR = Path("files")
MODEL = "gpt-4o"

print(f"Dataframer SDK version: {dataframer.__version__}")

Dataframer SDK version: 0.7.0


## Read Samples

Each sample folder contains one bank statement PDF and a `qa-pairs.csv` with question/answer pairs.

In [5]:
sample_dirs = sorted([d for d in FILES_DIR.iterdir() if d.is_dir()])

samples = []
for sample_dir in sample_dirs:
    pdf_paths = list(sample_dir.glob("*.pdf"))
    qa_path = sample_dir / "qa-pairs.csv"

    pdf_path = pdf_paths[0] if pdf_paths else None
    qa_df = pd.read_csv(qa_path) if qa_path.exists() else None

    samples.append({
        "dir": sample_dir,
        "pdf_path": pdf_path,
        "qa_path": qa_path,
        "qa_df": qa_df,
    })

    pdf_name = pdf_path.name if pdf_path else "(no PDF)"
    q_count = len(qa_df) if qa_df is not None else 0
    print(f"  {sample_dir.name}: {pdf_name} | {q_count} Q&A pairs")

print(f"\nLoaded {len(samples)} samples")

  sample1: BANK OF AMERICA.pdf | 5 Q&A pairs
  sample2: MORGAN STANLEY.pdf | 5 Q&A pairs
  sample3: JP MORGAN.pdf | 5 Q&A pairs
  sample4: CATHAY BANK.pdf | 5 Q&A pairs

Loaded 4 samples


## Create Multi-Folder Seed Dataset

All sample folders are packed into a ZIP preserving the folder hierarchy and uploaded to DataFramer. The folder structure (`sample1/`, `sample2/`, …) causes DataFramer to treat this as a **multi-folder** dataset.

In [6]:
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for sample in samples:
        for file_path in sorted(sample["dir"].iterdir()):
            arcname = f"{sample['dir'].name}/{file_path.name}"
            zf.write(file_path, arcname=arcname)
            print(f"  Added: {arcname}")
zip_buffer.seek(0)

dataset = df_client.dataframer.seed_datasets.create_from_zip(
    name=f"bank_statements_golden_labels_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    description="Bank financial statement PDFs with golden Q&A pairs — multi-folder seed dataset",
    zip_file=zip_buffer,
)

dataset_id = dataset.id
print(f"\nUpload complete")
print(f"Dataset ID   : {dataset_id}")
print(f"Dataset type : {dataset.dataset_type}")
print(f"Files        : {dataset.file_count}")
print(f"Folders      : {dataset.folder_count}")

  Added: sample1/BANK OF AMERICA.pdf
  Added: sample1/qa-pairs.csv
  Added: sample2/MORGAN STANLEY.pdf
  Added: sample2/qa-pairs.csv
  Added: sample3/JP MORGAN.pdf
  Added: sample3/qa-pairs.csv
  Added: sample4/CATHAY BANK.pdf
  Added: sample4/qa-pairs.csv

Upload complete
Dataset ID   : c6bf550f-4771-432b-ac0f-91a55c24dbde
Dataset type : MULTI_FOLDER
Files        : 8
Folders      : 4


## Generate a Specification

DataFramer analyzes the seed folders and produces a spec. The `generation_objectives` explain that `qa-pairs.csv` contains questions and their golden answers for the bank statement PDF co-located in the same sample folder.

In [26]:
# Set to an existing spec ID to skip spec creation and reuse a previous spec.
# Leave as None to generate a new spec from the dataset above.
EXISTING_SPEC_ID = None  # e.g. "3e5efcbb-60e1-4ddc-849c-6a192a44abe5"
NEW_DATA_PROPERTY='Bank Profile'

In [ ]:
if EXISTING_SPEC_ID:
    spec_id = EXISTING_SPEC_ID
    print(f"Reusing existing spec: {spec_id}")
else:
    spec_name = f"bank_statements_golden_labels_spec_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    spec = df_client.dataframer.specs.create(
        dataset_id=dataset_id,
        name=spec_name,
        spec_generation_model_name="anthropic/claude-opus-4-6-thinking",  # Check out available models on docs.dataframer.ai
        generation_objectives=(
            f"Do include {NEW_DATA_PROPERTY} "
            "as a data property. "
            "As an example, a value could be 'CRE-heavy community bank with Total CRE >300% of Tier 1'"
        ),
        extrapolate_values=False,
        generate_distributions=True,
    )

    spec_id = spec.id
    print(f"Started spec analysis: {spec_name}")
    print(f"Spec ID: {spec_id}")

    def spec_not_ready(result):
        return result.status not in ("SUCCEEDED", "FAILED")

    @retry(wait=wait_fixed(5), retry=retry_if_result(spec_not_ready), stop=stop_never)
    def poll_spec_status(spec_id):
        return df_client.dataframer.specs.retrieve(spec_id=spec_id)

    print("Polling for spec status...")
    spec_result = poll_spec_status(spec_id)

    if spec_result.status == "FAILED":
        raise RuntimeError("Spec analysis failed")

    print(f"\nSpec analysis completed successfully!")
    print(f"Spec ID: {spec_id}")

Reusing existing spec: d6995e6b-84c0-4de4-ad37-d34aef628955


## Review the Generated Specification

In [24]:
import textwrap

spec = df_client.dataframer.specs.retrieve(spec_id=spec_id)
config = yaml.safe_load(spec.content_yaml)
spec_data = config.get("spec", config)

print(f"Spec YAML length: {len(spec.content_yaml):,} chars")
print()

print(textwrap.fill(spec_data["requirements"]))

if "data_property_variations" in spec_data:
    print("Data properties:")
    for prop in spec_data["data_property_variations"]:
        print(f"  - {prop['property_name']}: {len(prop['property_values'])} values")

print()
print("Full spec YAML:")
print(spec.content_yaml)

Spec YAML length: 27,910 chars

Each data point is a multi-file collection of exactly 2 files: (1) a
file named 'qa-pairs.csv' containing exactly 5 question-answer pairs
in CSV format with columns 'question' and 'answer', and (2) a PDF file
named '[BANK_NAME].pdf' rendered as a single-page HTML document
representing a UBPR (Uniform Bank Performance Report) 'Analysis of
Concentrations of Credit--Page 7B' report from the FFIEC Central Data
Repository. The 5 questions in the CSV are always identical across all
samples: Q1 asks for total commercial real estate loans as a
percentage of Tier 1 Capital plus ACL (a single number), Q2 asks for
the bank's NOO CRE 3-year growth rate (a single number or 'N/A'), Q3
asks for the percentage share of the largest single loan category in
the total loan portfolio, Q4 asks for the bank's 1-4 family
residential exposure as a percentage of Tier 1 Capital, and Q5 asks
whether the bank's non-depository and other loans category is growing
faster than its Tier 

In [27]:
TARGET_PROPERTIES = [
    NEW_DATA_PROPERTY,
]

for prop in spec_data.get("data_property_variations", []):
    if prop["property_name"] in TARGET_PROPERTIES:
        print(f"{prop['property_name']}")
        for v in prop["property_values"]:
            print(f"  * {v}")
        print()

Bank Profile
  * CRE-heavy community bank with Total CRE >300% of Tier 1
  * Wealth management private bank with dominant securities-based and residential lending
  * Mono-line residential mortgage bank with near-100% concentration in 1-4 family
  * Diversified large bank with moderate balanced exposure across all categories
  * C&I-focused commercial bank with limited real estate exposure
  * Multifamily-focused bank with significant apartment lending concentration
  * Consumer lending-focused bank with dominant credit card and auto exposure
  * Agricultural community bank with farmland and ag loan concentration
  * SBA and small business-focused community bank with owner-occupied CRE



## Edit the Specification

We append a strict numerical consistency requirement to the shared requirements, then override the distribution for the *Bank Profile* property so that the value **"CRE-heavy community bank with Total CRE >300% of Tier 1"** carries **100%** weight and all other values carry **0%**.

In [ ]:
TARGET_BANK_PROFILE = "CRE-heavy community bank with Total CRE >300% of Tier 1"

CONSISTENCY_REQUIREMENT = """
Very important: the document must be 100% internally consistent. All generated numbers must be consistent with each other. You MUST use the calculator tool to verify every single numerical relationship in the document for every number.

Before using the calculator, first plan the complete structure of the document: every table, every row label, every column, every entity/location. Decide what line items exist and how they roll up into subtotals and totals. Only after that create SINGLE, COMPLETE script with ALL calculations included in that one script, using the calculator to verify/derive or compute every single number that will appear in the document. Specifically:
- This applies across ALL columns and ALL rows of ALL tables, everything must be verified in ONE script.
- You are not allowed to rely solely on mental arithmetic for any calculation. If a number is to appear in the document, the script must confirm or compute it.
- If the document contains multiple locations, entities, departments, or time periods, the calculator must compute numbers for ALL of them — not just one as a template.
- A couple of examples of required calculations: Every subtotal and total must be verified as the exact sum of its component line items. Every percentage must be verified as the correct division, rounded to the displayed precision. Every cross-entity consolidation (e.g. summing locations, departments, or facilities) must be verified for every row.

There is NO tolerance for even small errors, arithmetic operations, it's 0. This means if sum or % of whatever aggregate deviates from its correct value even by $0.01, this is a grave problem that can't be there after sample is generated.

If you really need to recompute some things even after the script execution, you should call the calculator again for those things. If you are already satisfied with execution results, do not call the calculator again, use what you got.

Additionally, every answer in qa-pairs.csv MUST be exactly derivable from the numbers present in the accompanying PDF. Verify each Q&A pair against the PDF content before finalising the sample.
"""

updated_spec_data = copy.deepcopy(spec_data)

existing_requirements = updated_spec_data.get("requirements") or ""
updated_spec_data["requirements"] = existing_requirements + CONSISTENCY_REQUIREMENT

for prop in updated_spec_data["data_property_variations"]:
    if prop["property_name"] == NEW_DATA_PROPERTY:
        prop["base_distributions"] = {
            v: (100 if v == TARGET_BANK_PROFILE else 0)
            for v in prop["property_values"]
        }
        print(f"Overrode distribution for: {prop['property_name']}")
        print(f"  Values: {prop['property_values']}")
        print(f"  Distributions: {prop['base_distributions']}")

updated_yaml = yaml.dump({"spec": updated_spec_data}, allow_unicode=True, sort_keys=False)

updated_spec = df_client.dataframer.specs.update(spec_id=spec_id, content_yaml=updated_yaml)

print(f"\nSpec updated")
print(f"Spec ID: {updated_spec.id}")
print(f"Status:  {updated_spec.status}")

## Create a Generation Run

Using the updated spec, we launch a run that generates **5 samples** with conformance revision, conformance filtering, a 16 000-token thinking budget, and the calculator tool.

In [ ]:
# Set to an existing run ID to skip run creation and jump straight to download.
# Leave as None to launch a new generation run using the updated spec above.
EXISTING_RUN_ID = None  # e.g. "77ddd31a-2c5a-448b-b00f-a157f42f2de8"

In [32]:
if EXISTING_RUN_ID:
    run_id = EXISTING_RUN_ID
    print(f"Reusing existing run: {run_id}")
else:
    run = df_client.dataframer.runs.create(
        spec_id=updated_spec.id,
        number_of_samples=3,
        generation_model="anthropic/claude-opus-4-6-thinking",
        revision_model="anthropic/claude-opus-4-6-thinking",
        revision_types=["conformance"],
        max_revision_cycles=1,
        filtering_types=["conformance"],
        generation_thinking_budget=16000,
        tools=["calculator"],
        skip_outline=False,
        unified_multifield=False
    )

    run_id = run.id
    print(f"Run started")
    print(f"Run ID: {run_id}")

    def run_not_ready(result):
        return result.status not in ("SUCCEEDED", "FAILED")

    @retry(wait=wait_fixed(10), retry=retry_if_result(run_not_ready), stop=stop_never)
    def poll_run_status(run_id):
        return df_client.dataframer.runs.retrieve(run_id=run_id)

    print("Polling for run status...")
    run_result = poll_run_status(run_id)

    if run_result.status == "FAILED":
        raise RuntimeError("Run failed")

    print(f"\nRun completed successfully!")
    print(f"Run ID:            {run_id}")
    print(f"Samples completed: {run_result.samples_completed}")
    print(f"Samples failed:    {run_result.samples_failed}")

Reusing existing run: d70b4f31-8e36-43bc-a17b-14d455cc5e68


## Download Generated Samples

In [ ]:
base_url = str(df_client.base_url).rstrip("/")
cost_data = requests.get(
    f"{base_url}/api/dataframer/runs/{run_id}/cost/",
    headers={"Authorization": f"Bearer {os.environ['DATAFRAMER_API_KEY']}"},
).json()

total_cost = cost_data.get("total_cost_cents")
print(f"Total cost for all samples: ${total_cost / 100:.2f}")


def zip_not_ready(result):
    return result.download_url is None


@retry(wait=wait_fixed(5), retry=retry_if_result(zip_not_ready), stop=stop_never)
def poll_download_all(run_id):
    return df_client.dataframer.runs.files.download_all(run_id=run_id)


print("\nGenerating download ZIP...")
dl = poll_download_all(run_id)
print(f"ZIP ready — {dl.size_bytes:,} bytes")

output_dir = Path("output") / run_id
output_dir.mkdir(parents=True, exist_ok=True)

zip_data = requests.get(dl.download_url).content
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extractall(output_dir)
    names = sorted(zf.namelist())

print(f"\nExtracted {len(names)} files to {output_dir}/")
for name in names:
    print(f"  {name}")

## Inspect a Generated Sample

Print the Q&A pairs from the first generated sample, then display the DataFramer annotation tags from the run metadata to confirm the spec properties were applied as configured.

In [ ]:
import json
import textwrap

first_sample_dir = sorted([p for p in output_dir.iterdir() if p.is_dir()])[0]
print(f"Inspecting: {first_sample_dir}")

print(f"\n=== Q&A pairs for {first_sample_dir.name} ===")
qa_df = pd.read_csv(first_sample_dir / "qa-pairs.csv")
for _, row in qa_df.iterrows():
    print(f"  Q: {textwrap.fill(str(row['question']), width=100, subsequent_indent='     ')}")
    print(f"  A: {row['answer']}")
    print()

print(f"=== DataFramer generation tags for {first_sample_dir.name} ===")
with open(first_sample_dir / "folder.metadata") as f:
    folder_meta = json.load(f)

for key, value in folder_meta.get("generation_tags", {}).items():
    wrapped_value = textwrap.fill(str(value), width=100, subsequent_indent="     ")
    print(f"  {key}:\n     {wrapped_value}")
    print()

## Human Review

Before running automated evaluation, review the generated Q&A pairs and their source PDFs to verify the labels are accurate and well-formed.

## Evaluate First Generated Sample

Read the first sample's PDF and `qa-pairs.csv`, then ask OpenAI each question and compare the model answer to the golden label.

In [ ]:
# Find the first sample folder in the output directory
sample_folders = sorted([p for p in output_dir.iterdir() if p.is_dir()])
first_sample = sample_folders[0]
print(f"First sample folder: {first_sample}")

first_pdf = next(first_sample.glob("*.pdf"), None)
first_qa = first_sample / "qa-pairs.csv"

print(f"PDF       : {first_pdf}")
print(f"QA pairs  : {first_qa}")

# Extract PDF text
doc = fitz.open(first_pdf)
pdf_text = "\n\n".join(page.get_text() for page in doc)
print(f"\nExtracted {len(pdf_text):,} chars from PDF")

# Load Q&A pairs
qa_df = pd.read_csv(first_qa)
print(f"Q&A pairs loaded: {len(qa_df)} rows")
print()
print(qa_df.to_string(index=False))

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise financial data extraction assistant. "
    "The user will provide the full text of a bank financial statement and ask a specific question. "
    "Answer with only the single requested value — no explanation, no units label, no extra text."
    "Important - If answering with a number, answer with 2 decimal places."
)


def ask_question(pdf_text: str, question: str) -> str:
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Financial statement:\n\n{pdf_text}\n\nQuestion: {question}",
            },
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()


print(f"Evaluating {len(qa_df)} questions against generated PDF...\n")

eval_results = []
for _, row in qa_df.iterrows():
    question = row["question"]
    golden = str(row["answer"])
    model_answer = ask_question(pdf_text, question)
    match = model_answer == golden
    eval_results.append({
        "question": question,
        "model_answer": model_answer,
        "golden": golden,
        "exact_match": match,
    })
    status = "✓" if match else "✗"
    print(f"[{status}] model={model_answer!r}  golden={golden!r}")
    print(f"    Q: {question[:100]}")
    print()

results_df = pd.DataFrame(eval_results)
accuracy = results_df["exact_match"].mean()
print(f"Exact-match accuracy: {accuracy:.1%} ({int(results_df['exact_match'].sum())}/{len(results_df)})")